# KRIOS GIS — Fingrid Grid Capacity Layer Check

This notebook discovers and validates the **Fingrid substation consumption capacity** dataset — the authoritative source for the assignment's "Sähkön kulutuskapasiteetti" (electricity consumption capacity) requirement.

### Layer lineage

| Grid Scope web map | Assignment reference | ArcGIS FeatureServer |
|---|---|---|
| `Sähkön / kulutuskapasiteetti` (group) | "use Sähkön kulutuskapasiteetti layer" | — |
| `Substations` (active layer inside group) | The actual data layer | `Kytkinlaitokset_Fingrid` → `Sähköasemat_liityntäkapasiteetti` |

**Key field:** `Kulutus_25` = consumption headroom (MW) per substation

**Source:** [Grid Scope map](https://karttapalaute.fingrid.fi/?setlanguage=en&?link=3opMB) — Fingrid's interactive map for grid connection capacity estimates.

## Workflow

1. **Chunk 1** -- List all FeatureServer services from the ArcGIS host; confirm Kytkinlaitokset_Fingrid exists.
2. **Chunk 2** -- Drill into the layer, show its schema, field definitions, and a sample of attribute values.
3. **Chunk 2b** -- Download ALL Fingrid substations with geometry, save to ./data/fingrid_substations.geojson.
4. **Chunk 3** -- Extract OSM layers (power lines, substations, power plants, data centers) via single Overpass union, save to ./data/.
5. **Chunk 3b** -- Extract urban centers (population >= 100k) from OSM, save to ./data/urban_centers.geojson.
6. **Chunk 3c** -- Extract land parcels >= 10 ha from MML API, save to ./data/land_parcels.geojson. Requires MML key.
7. **Chunk 3d** -- Download DEM (2m) via MML, compute slope, reclassify < 8%, save to ./data/. Requires MML key + GDAL.
8. **Chunk 3e** -- Extract exclusion zones (Natura 2000, flood maps, nature reserves) from EEA + SYKE, save to ./data/exclusion_zones.geojson.
9. **Chunk 4** -- Combined map: all 8 layers on a single GeoLibre widget.


In [9]:
import pandas as pd
import requests

# ArcGIS REST endpoint hosting Fingrid's geospatial data
BASE_URL = "https://services2.arcgis.com/uh3cDCipmuPcmxmx/ArcGIS/rest/services"

# Chunk 1: list all native ArcGIS services and keep FeatureServer items.
root_resp = requests.get(f"{BASE_URL}?f=pjson", timeout=45)
print("Root status:", root_resp.status_code)
root_resp.raise_for_status()
root_json = root_resp.json()

services = root_json.get("services", [])
services_df = pd.DataFrame(services)

if services_df.empty:
    raise RuntimeError("No services returned from ArcGIS root endpoint.")

services_df["service_url"] = (
    BASE_URL + "/" + services_df["name"].astype(str) + "/" + services_df["type"].astype(str)
    )
feature_services_df = services_df[services_df["type"].eq("FeatureServer")].copy()

print("Total services:", len(services_df))
print("FeatureServer count:", len(feature_services_df))
display(feature_services_df[["name", "type", "service_url"]].sort_values("name").reset_index(drop=True))

Root status: 200
Total services: 113
FeatureServer count: 113


,name,type,service_url
0,Asemakaavoitettualue_VESISTOALUEET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
1,Asuinrakennus,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
2,Biokaasu,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
3,EVO_OYK,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
4,EVO_OYK_KIINTEISTÖTUNNUKSET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
...,...,...,...
108,survey123_af37f13c356e4d2581a14b302ed42ec9_res...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
109,survey123_d8fbac0dec194234a9b66a751e1c8b8a_form,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
110,survey123_f9551175b74545e8a56f137d280efef9_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
111,survey123_fa3c5b79a00e48b88cf0e75cbd493ac0_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...


In [10]:
# Chunk 2: extract the Fingrid substation connection capacity layer and inspect its schema + sample data.
# Assignment: "Grid capacity & headroom — Fingrid — use Sähkön kulutuskapasiteetti layer"
# Published as: Kytkinlaitokset_Fingrid -> Sähköasemat_liityntäkapasiteetti
# Key field: Kulutus_25 = consumption capacity headroom (MW)

TARGET_SERVICE_NAME = "Kytkinlaitokset_Fingrid"
TARGET_LAYER_HINT = "liityntäkapasiteetti"  # matches Sähköasemat_liityntäkapasiteetti

# Locate the target service
target_service = feature_services_df[feature_services_df["name"] == TARGET_SERVICE_NAME].copy()
if target_service.empty:
    raise RuntimeError(f"Target service not found: {TARGET_SERVICE_NAME}")

service_url = target_service.iloc[0]["service_url"]
service_resp = requests.get(f"{service_url}?f=pjson", timeout=45)
print("Service status:", service_resp.status_code)
service_resp.raise_for_status()
service_json = service_resp.json()

layers = service_json.get("layers", [])
layers_df = pd.DataFrame(layers)
if layers_df.empty:
    raise RuntimeError("No layers found inside target FeatureServer.")

# Find the substation connection capacity layer by name hint
layers_df["name_lc"] = layers_df["name"].astype(str).str.lower()
layer_match = layers_df[layers_df["name_lc"].str.contains(TARGET_LAYER_HINT.lower(), na=False)].copy()
selected_layer = layer_match.iloc[0] if not layer_match.empty else layers_df.iloc[0]

layer_url = f"{service_url}/{int(selected_layer['id'])}"
print("Selected layer:", selected_layer["name"])
print("Layer URL:", layer_url)

# Fetch layer metadata
meta_resp = requests.get(f"{layer_url}?f=pjson", timeout=45)
print("Layer metadata status:", meta_resp.status_code)
meta_resp.raise_for_status()
meta = meta_resp.json()

meta_summary = {
    "name": meta.get("name"),
    "type": meta.get("type"),
    "geometryType": meta.get("geometryType"),
    "objectIdField": meta.get("objectIdField"),
    "displayField": meta.get("displayField"),
    "maxRecordCount": meta.get("maxRecordCount"),
}
print("\nMetadata summary:", meta_summary)

# Show field definitions
fields_df = pd.DataFrame(meta.get("fields", []))
print("\nField count:", len(fields_df))
if not fields_df.empty:
    cols = [c for c in ["name", "alias", "type", "length", "nullable"] if c in fields_df.columns]
    display(fields_df[cols].sort_values("name").reset_index(drop=True))

# Query 10 sample records (no geometry, compact)
query_url = f"{layer_url}/query"
params = {
    "f": "json",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 10,
}
sample_resp = requests.get(query_url, params=params, timeout=45)
print("\nSample query status:", sample_resp.status_code)
sample_resp.raise_for_status()
sample_json = sample_resp.json()

rows = [f.get("attributes", {}) for f in sample_json.get("features", [])]
sample_df = pd.DataFrame(rows)
print("Sample row count:", len(sample_df))
if not sample_df.empty:
    preferred_cols = [c for c in ["SA", "Tyyppi", "Kulutus_25", "Tuotanto__", "Tuotanto_1", "Tuotanto_2", "X", "Y"] if c in sample_df.columns]
    display(sample_df[preferred_cols] if preferred_cols else sample_df.head(10))

Service status: 200
Selected layer: Sähköasemat_liityntäkapasiteetti
Layer URL: https://services2.arcgis.com/uh3cDCipmuPcmxmx/ArcGIS/rest/services/Kytkinlaitokset_Fingrid/FeatureServer/0
Layer metadata status: 200

Metadata summary: {'name': 'Sähköasemat_liityntäkapasiteetti', 'type': 'Feature Layer', 'geometryType': 'esriGeometryPoint', 'objectIdField': 'FID', 'displayField': 'SA', 'maxRecordCount': 2000}

Field count: 9


,name,alias,type,length,nullable
0,FID,FID,esriFieldTypeOID,NaN,False
1,Kulutus_25,Kulutus_25,esriFieldTypeInteger,NaN,True
2,SA,SA,esriFieldTypeString,254.0,True
3,Tuotanto_1,Tuotanto_1,esriFieldTypeInteger,NaN,True
4,Tuotanto_2,Tuotanto_2,esriFieldTypeInteger,NaN,True
5,Tuotanto__,Tuotanto__,esriFieldTypeInteger,NaN,True
6,Tyyppi,Tyyppi,esriFieldTypeString,254.0,True
7,X,X,esriFieldTypeDouble,NaN,True
8,Y,Y,esriFieldTypeDouble,NaN,True



Sample query status: 200
Sample row count: 10


,SA,Tyyppi,Kulutus_25,Tuotanto__,Tuotanto_1,Tuotanto_2,X,Y
0,AHVENKOSKI 110 kV,110,0,200,200,200,469786.110,6706431.649
1,ALAJÄRVI 110 kV,110,500,60,60,60,356668.873,6992357.131
2,ALAJÄRVI 400 kV,400,500,0,0,0,356668.873,6992357.131
3,ALAPITKÄ 110 kV,110,200,180,180,180,526241.848,7006646.487
4,ALAPITKÄ 400 kV,400,500,380,380,380,526241.848,7006646.487
5,ANTTILA 110 kV,110,0,500,500,500,409988.222,6694581.005
6,ANTTILA 400 kV,400,0,500,500,500,409988.222,6694581.005
7,ARKKUKALLIO 110 kV,110,500,0,240,240,222165.836,6898752.332
8,ARKKUKALLIO 400 kV,400,500,0,440,440,222165.836,6898752.332
9,ESPOO 110 kV,110,0,500,500,500,364250.795,6676726.023


In [11]:
# Chunk 2b: download ALL Fingrid substations with geometry and save to GeoJSON
# Uses the layer_url from Chunk 2 (must be run first)

from pathlib import Path
import json
import math
import time

OUT = Path("./data")
OUT.mkdir(parents=True, exist_ok=True)

# ArcGIS query: paginate through all records with geometry
def query_arcgis_all(url, where="1=1", out_fields="*", out_sr=4326, page_size=2000):
    """Fetch all features from an ArcGIS FeatureServer layer with pagination."""
    all_features = []
    offset = 0
    while True:
        params = {
            "f": "json",
            "where": where,
            "outFields": out_fields,
            "returnGeometry": "true",
            "outSR": out_sr,
            "resultRecordCount": page_size,
            "resultOffset": offset,
        }
        r = requests.get(f"{layer_url}/query", params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        features = data.get("features", [])
        if not features:
            break
        all_features.extend(features)
        offset += page_size
        time.sleep(0.5)
    return all_features

print("Downloading all Fingrid substations...")
raw_features = query_arcgis_all(layer_url, out_sr=4326)
print(f"Fetched {len(raw_features)} features")

# Convert ArcGIS JSON to GeoJSON
geojson_features = []
for f in raw_features:
    attrs = f.get("attributes", {})
    geom = f.get("geometry")
    if geom and "x" in geom and "y" in geom:
        feature = {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [geom["x"], geom["y"]]},
            "properties": {
                "name": attrs.get("SA", ""),
                "voltage_kv": attrs.get("Tyyppi", None),
                "consumption_mw": attrs.get("Kulutus_25", None),
                "production_1_mw": attrs.get("Tuotanto_1", None),
                "production_2_mw": attrs.get("Tuotanto_2", None),
                "production_total_mw": attrs.get("Tuotanto__", None),
            }
        }
        feature["properties"] = {k: v for k, v in feature["properties"].items() if v is not None}
        geojson_features.append(feature)

fc = {"type": "FeatureCollection", "features": geojson_features}
out_path = OUT / "fingrid_substations.geojson"
out_path.write_text(json.dumps(fc, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(geojson_features)} features to {out_path}")


Fetched 172 features
Saved 172 features to data\fingrid_substations.geojson


In [12]:
# Chunk 3: extract ALL OSM layers in a single Overpass query (faster, fewer requests)
# AOI: Helsinki region (~100km radius: 59.7-60.7 lat, 23.9-26.0 lon)

import json
from pathlib import Path
import requests
import time

UA = "KRIOS-GIS-Assignment/1.0 (research)"
OSM_BBOX = "59.7,23.9,60.7,26.0"
OUT = Path("./data")
OUT.mkdir(parents=True, exist_ok=True)

def overpass_query(query, label, retries=3):
    """Single Overpass query with retry on 429."""
    for attempt in range(retries + 1):
        r = requests.post(
            "https://overpass-api.de/api/interpreter",
            data={"data": query},
            headers={"User-Agent": UA},
            timeout=120
        )
        if r.status_code == 429 and attempt < retries:
            wait = 10 * (2 ** attempt)
            print(f"Rate limited ({label}), retrying in {wait}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Failed after {retries} retries: {label}")

def elements_to_geojson(elements, label):
    """Convert Overpass elements list to GeoJSON FeatureCollection and save."""
    features = []
    for el in elements:
        geom = None
        if el["type"] == "node":
            geom = {"type": "Point", "coordinates": [el["lon"], el["lat"]]}
        elif el["type"] == "way" and "geometry" in el:
            coords = [[p["lon"], p["lat"]] for p in el["geometry"]]
            if len(coords) == 1:
                geom = {"type": "Point", "coordinates": coords[0]}
            else:
                geom = {"type": "LineString", "coordinates": coords}
        if geom:
            features.append({
                "type": "Feature",
                "geometry": geom,
                "properties": el.get("tags", {})
            })
    fc = {"type": "FeatureCollection", "features": features}
    safe = label.lower().replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
    path = OUT / f"osm_{safe}.geojson"
    path.write_text(json.dumps(fc, ensure_ascii=False), encoding="utf-8")
    print(f"{label}: {len(features)} features -> {path}")
    return fc

# Single Overpass query combining all 4 datasets
# Uses union block: ( ... ; ... ; ... ; ) to get everything in one call
q = (
    f'[out:json][timeout:90][bbox:{OSM_BBOX}];'
    '('
    '  way["power"="line"]["voltage"~"400000|220000|110000"];'
    '  node["power"="substation"];'
    '  way["power"="substation"];'
    '  node["power"="plant"];'
    '  way["power"="plant"];'
    '  node["power"="generator"];'
    '  way["power"="generator"];'
    '  node["datacenter"~"yes"];'
    '  way["datacenter"~"yes"];'
    '  node["building"="data_center"];'
    '  way["building"="data_center"];'
    ');'
    'out geom;'
)

data = overpass_query(q, "OSM all layers")
elements = data.get("elements", [])
print(f"Total OSM elements fetched: {len(elements)}")

# Split by tag into 4 GeoJSONs
lines_el = [e for e in elements if e.get("tags", {}).get("power") == "line"]
subs_el = [e for e in elements if e.get("tags", {}).get("power") == "substation"]
plants_el = [e for e in elements if e.get("tags", {}).get("power") in ("plant", "generator")]
dc_el = [e for e in elements if e.get("tags", {}).get("datacenter") == "yes" or e.get("tags", {}).get("building") == "data_center"]

lines = elements_to_geojson(lines_el, "Power lines (110-400 kV)")
substations = elements_to_geojson(subs_el, "Substations (OSM)")
plants = elements_to_geojson(plants_el, "Power plants/generators")
datacenters = elements_to_geojson(dc_el, "Data centers")


Total OSM elements fetched: 2030
Power lines (110-400 kV): 422 features -> data\osm_power_lines_110-400_kv.geojson
Substations (OSM): 858 features -> data\osm_substations_osm.geojson
Power plants/generators: 750 features -> data\osm_power_plants_generators.geojson
Data centers: 0 features -> data\osm_data_centers.geojson


In [ ]:
# Chunk 3b: urban centers (population >= 100k) from OSM
# Proximity to large cities = workforce availability signal for MCDA

# Query OSM for Finnish cities/towns with population >= 100,000
# Using tighter bbox to exclude Russian cities
bbox_finland = "59.7,20.5,70.5,30.0"
q = (
    f'[out:json][timeout:60][bbox:{bbox_finland}];'
    '('
    '  node["place"~"city|town"][population];'
    '  way["place"~"city|town"][population];'
    ');'
    'out center;'
)

r = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={"data": q},
    headers={"User-Agent": UA},
    timeout=60
)
r.raise_for_status()
data = r.json()
elements = data.get("elements", [])

cities = []
for el in elements:
    tags = el.get("tags", {})
    # Skip non-Finnish places from bbox overlap
    country = tags.get("is_in:country", "") or tags.get("addr:country", "")
    if country and country.lower() not in ("fi", "finland", "suomi", ""):
        continue
    pop_str = str(tags.get("population", "0")).replace(",", "").replace(" ", "")
    try:
        pop = int(pop_str)
    except:
        continue
    if pop >= 100000:
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        cities.append({
            "name": tags.get("name", "?"),
            "population": pop,
            "lat": lat, "lon": lon
        })

print(f"Finnish urban centers >= 100k: {len(cities)}")
for c in sorted(cities, key=lambda x: -x["population"]):
    print(f"  {c['name']}: {c['population']:,}")

# Save as GeoJSON
features = []
for c in cities:
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [c["lon"], c["lat"]]},
        "properties": {"name": c["name"], "population": c["population"]}
    })
urban_centers = {"type": "FeatureCollection", "features": features}
path = Path("./data/urban_centers.geojson")
path.write_text(json.dumps(urban_centers, ensure_ascii=False), encoding="utf-8")
print(f"Saved to {path}")


In [ ]:
# Chunk 3c: land parcels >= 10 ha from MML OGC API Features
# Uses MML cadastral index map, computes area from polygon geometry, filters >= 10 ha
# Requires MML API key (free): https://omatili.maanmittauslaitos.fi/user/new/avoimet-rajapintapalvelut?lang=en

import json
import time
from pathlib import Path
from shapely.geometry import shape
import requests

MML_KEY = ""  # <-- SET YOUR MML API KEY HERE
OUT = Path("./data")
OUT.mkdir(parents=True, exist_ok=True)

# MML OGC API Features endpoint for parcels
BASE = "https://avoin-paikkatieto.maanmittauslaitos.fi/kiinteisto-avoin/simple-features/v3"

# Search multiple areas around Helsinki for parcels >= 10 ha (EPSG:3067)
# These bboxes cover suburban/rural fringe where large parcels exist
SEARCH_BBOXES = [
    "385000,6680000,392000,6685000",  # Vantaa north industrial
    "405000,6680000,425000,6695000",  # Sipoo rural corridor
    "360000,6670000,375000,6690000",  # Kirkkonummi east
    "380000,6695000,400000,6710000",  # Nurmijärvi-Tuusula agricultural
    "410000,6675000,430000,6690000",  # Porvoo west corridor
    "355000,6690000,375000,6710000",  # Vihti-Lohja west
]

all_large = []
seen_ids = set()

for bbox in SEARCH_BBOXES:
    url = (
        f"{BASE}/collections/PalstanSijaintitiedot/items"
        f"?bbox={bbox}"
        f"&bbox-crs=http://www.opengis.net/def/crs/EPSG/0/3067"
        f"&crs=http://www.opengis.net/def/crs/EPSG/0/3067"
        f"&limit=1000"
    )
    count = 0
    while url and count < 2000:
        r = requests.get(url, auth=(MML_KEY, ""), timeout=120)
        if r.status_code != 200:
            break
        data = r.json()
        feats = data.get("features", [])
        for f in feats:
            g = f["geometry"]
            if g and g["type"] == "Polygon":
                try:
                    s = shape(g)
                    ha = s.area / 10000  # m² -> ha
                    pid = f["properties"].get("kiinteistotunnus", "")
                    if ha >= 10 and pid not in seen_ids:
                        seen_ids.add(pid)
                        all_large.append({
                            "type": "Feature", "geometry": g,
                            "properties": {"id": pid, "area_ha": round(ha, 1)}
                        })
                except:
                    pass
        count += len(feats)
        links = data.get("links", [])
        url = None
        for l in links:
            if l.get("rel") == "next":
                url = l["href"]
                break
        time.sleep(0.5)
    print(f"  Area done: {len(all_large)} large parcels so far")

land_parcels = {"type": "FeatureCollection", "features": all_large}
path = OUT / "land_parcels.geojson"
path.write_text(json.dumps(land_parcels, ensure_ascii=False), encoding="utf-8")
print(f"\nTotal parcels >= 10 ha: {len(all_large)} -> {path}")
for p in sorted(all_large, key=lambda x: -x["properties"]["area_ha"])[:5]:
    print(f"  {p['properties']['id']}: {p['properties']['area_ha']} ha")


In [ ]:
# Chunk 3d: DEM + gradient (< 8%) from MML
# Downloads 2m elevation model via MML OGC API Processes, computes slope with GDAL
# Requires: MML API key, GDAL installed (pip install gdal or OSGeo4W on Windows)

import json
import time
from pathlib import Path
import requests
import subprocess

MML_KEY = ""  # <-- SET YOUR MML API KEY HERE (same as layer 6)
OUT = Path("./data")
OUT.mkdir(parents=True, exist_ok=True)

# Step 1: Submit DEM job for Helsinki region via OGC API Processes
PROC = "https://avoin-paikkatieto.maanmittauslaitos.fi/tiedostopalvelu/ogcproc/v1/"
PID = "korkeusmalli_2m_bbox"

# Helsinki core area in EPSG:3067 (3km x 3km test)
# For full AOI, expand this bbox or loop multiple tiles
BBOX_DEM = [384000, 6669000, 400000, 6685000]  # ~16km x 16km

body = json.dumps({
    "id": PID,
    "inputs": {
        "boundingBoxInput": BBOX_DEM,
        "fileFormatInput": "TIFF"
    }
})
r = requests.post(f"{PROC}processes/{PID}/execution",
    data=body, auth=(MML_KEY, ""),
    timeout=30, headers={"Content-Type": "application/json"})
r.raise_for_status()
job_id = r.json().get("jobID")
print(f"DEM job submitted: {job_id}")

# Step 2: Poll until complete
for i in range(30):
    time.sleep(5)
    r2 = requests.get(f"{PROC}jobs/{job_id}", auth=(MML_KEY, ""), timeout=30)
    s = r2.json().get("status", "")
    print(f"  Poll {i+1}: {s}")
    if s == "successful":
        break
    if s in ("failed", "error"):
        raise RuntimeError(f"DEM job failed: {r2.json()}")
else:
    raise RuntimeError("DEM job timed out")

# Step 3: Download the GeoTIFF
res = requests.get(f"{PROC}jobs/{job_id}/results", auth=(MML_KEY, ""), timeout=30).json()
dl_url = None
for v in res.values():
    if isinstance(v, dict) and v.get("href"):
        dl_url = v["href"]
        break
r3 = requests.get(dl_url, auth=(MML_KEY, ""), timeout=600)
dem_path = OUT / "dem_2m_helsinki.tiff"
dem_path.write_bytes(r3.content)
print(f"DEM downloaded: {dem_path} ({len(r3.content)} bytes)")

# Step 4: Compute slope using GDAL
slope_path = OUT / "gradient_percent.tiff"
subprocess.run([
    "gdaldem", "slope", str(dem_path), str(slope_path),
    "-p",  # percent slope (not degrees)
    "-of", "GTiff"
], check=True)
print(f"Slope raster: {slope_path}")

# Step 5: Reclassify to binary: suitable (< 8%) = 1, unsuitable (>= 8%) = 0
suitable_path = OUT / "gradient_suitable_8pct.tiff"
subprocess.run([
    "gdal_calc.py",
    "-A", str(slope_path),
    "--outfile", str(suitable_path),
    "--calc", "1*(A<8)+0*(A>=8)",
    "--NoDataValue", "0",
    "--type", "Byte",
    "--overwrite"
], check=True)
print(f"Suitable gradient mask: {suitable_path}")
print("Done. Suitable areas = 1 (slope < 8%), Unsuitable = 0 (slope >= 8%)")


In [ ]:
# Chunk 3e: exclusion zones -- Natura 2000, flood hazards, nature reserves
# Fatal flaw layers for MCDA: areas where development is restricted

import json
import requests
import time
from pathlib import Path

OUT = Path("./data")
OUT.mkdir(parents=True, exist_ok=True)

# Helsinki region bbox in EPSG:3067 (SYKE) and EPSG:4326 (Natura)
BBOX_3067 = "300000,6660000,500000,6730000,urn:x-ogc:def:crs:EPSG:3067"
BBOX_4326 = "19,59.5,32,70.5"

def fetch_syke_wfs(type_name, label, bbox=BBOX_3067, max_feats=5000):
    """Fetch features from SYKE WFS with pagination."""
    base = "https://paikkatiedot.ymparisto.fi/geoserver"
    ws = type_name.split(":")[0]
    url = f"{base}/{ws}/wfs"
    all_feats = []
    start_idx = 0
    while True:
        r = requests.get(url, params={
            "service": "WFS", "version": "2.0.0",
            "request": "GetFeature",
            "typeNames": type_name,
            "outputFormat": "application/json",
            "bbox": bbox,
            "count": 1000,
            "startIndex": start_idx
        }, timeout=120)
        if r.status_code != 200:
            break
        feats = r.json().get("features", [])
        if not feats:
            break
        all_feats.extend(feats)
        start_idx += len(feats)
        if len(feats) < 1000:
            break
        time.sleep(0.5)
    print(f"  {label}: {len(all_feats)} features")
    return all_feats

all_exclusion = []

# 1. Flood hazard zones (1-in-100 year, river + sea)
print("=== EXCLUSION ZONES ===")
for tn in [
    "inspire_nz:NZ.Tulvavaaravyohykkeet_Vesistotulva_1_100a",
    "inspire_nz:NZ.Tulvavaaravyohykkeet_Meritulva_1_100a",
]:
    label = "River flood 100a" if "Vesistot" in tn else "Sea flood 100a"
    feats = fetch_syke_wfs(tn, label)
    for f in feats:
        f["properties"]["hazard_type"] = "flood"
        f["properties"]["source"] = label
    all_exclusion.extend(feats)

# 2. Nature reserves (state-owned + Natura 2000 via SYKE)
for tn in [
    "inspire_ps:PS.ProtectedSitesValtionOmistamaLuonnonsuojelualue",
    "inspire_ps:PS.ProtectedSitesSpecialProtectionArea",  # Birds Directive
    "inspire_ps:PS.ProtectedSitesSpecialAreaOfConservation",  # Habitats Directive
]:
    label = tn.split(":")[-1][:30]
    feats = fetch_syke_wfs(tn, label)
    for f in feats:
        f["properties"]["hazard_type"] = "protected"
        f["properties"]["source"] = "nature_reserve"
        f["properties"]["name"] = f["properties"].get("nimi", "")
    all_exclusion.extend(feats)

# 3. Natura 2000 via EEA ArcGIS REST (all site types, Finnish sites only)
print("  Natura 2000 (EEA)...")
natura = "https://bio.discomap.eea.europa.eu/arcgis/rest/services/ProtectedSites/Natura2000Sites/MapServer/0"
offset = 0
while True:
    r = requests.get(f"{natura}/query", params={
        "f": "geojson",
        "where": "SITECODE LIKE 'FI%'",
        "outFields": "SITECODE,SITENAME,SITETYPE,Area_ha",
        "outSR": 4326,
        "returnGeometry": "true",
        "resultRecordCount": 1000,
        "resultOffset": offset
    }, timeout=120)
    if r.status_code != 200:
        break
    feats = r.json().get("features", [])
    if not feats:
        break
    for f in feats:
        f["properties"]["hazard_type"] = "protected"
        f["properties"]["source"] = "natura2000"
        f["properties"]["name"] = f["properties"].get("SITENAME", "")
    all_exclusion.extend(feats)
    offset += len(feats)
    if len(feats) < 1000:
        break
    time.sleep(0.5)
print(f"  Natura 2000: added")

# Save as single combined GeoJSON
excl = {"type": "FeatureCollection", "features": all_exclusion}
path = OUT / "exclusion_zones.geojson"
path.write_text(json.dumps(excl, ensure_ascii=False), encoding="utf-8")
print(f"Total exclusion features: {len(all_exclusion)} -> {path}")


In [15]:
# Chunk 4: combined map -- Fingrid + OSM layers on a single GeoLibre map

# Workaround for VS Code anywidget frontend issue: render/export HTML only (no widget output).
import json
from pathlib import Path
from IPython.display import HTML, display
from geolibre import Map

# Load Fingrid GeoJSON
fingrid_path = Path("./data/fingrid_substations.geojson")
fingrid = json.loads(fingrid_path.read_text(encoding="utf-8"))

# Load urban centers GeoJSON
urban_path = Path("./data/urban_centers.geojson")
urban_centers = json.loads(urban_path.read_text(encoding="utf-8"))

# Load land parcels GeoJSON (if exists)
parcels_path = Path("./data/land_parcels.geojson")
if parcels_path.exists():
    land_parcels = json.loads(parcels_path.read_text(encoding="utf-8"))

# Load exclusion zones GeoJSON (if exists)
excl_path = Path("./data/exclusion_zones.geojson")
if excl_path.exists():
    exclusion_zones = json.loads(excl_path.read_text(encoding="utf-8"))
    print(f"Loaded {len(exclusion_zones['features'])} exclusion zone features")
else:
    exclusion_zones = {"type": "FeatureCollection", "features": []}
    print("No exclusion zones data found")
    print(f"Loaded {len(land_parcels['features'])} land parcels")
else:
    land_parcels = {"type": "FeatureCollection", "features": []}
    print("No land parcels data found")
print(f"Loaded {len(urban_centers['features'])} urban centers")

# Create GeoLibre map centred on Helsinki region
m = Map(
    center=[25.0, 60.2],
    zoom=9,
    basemap="bright",
    layout="embed",
    height="700px",
)

# 1. Fingrid substations (consumption capacity)
m.add_choropleth(
    fingrid,
    column="consumption_mw",
    name="Fingrid Substations (consumption MW)",
    class_count=5,
    colormap="YlOrRd",
    scheme="quantile",
    type="circle",
    radius=10,
    stroke=True,
    popup=["name", "voltage_kv", "consumption_mw"],
)

# 2. OSM power lines
m.add_geojson(
    lines,
    name="Power lines (OSM)",
    color="#e74c3c",
    weight=2,
    popup=["voltage", "name", "operator"],
)

# 3. OSM substations
m.add_geojson(
    substations,
    name="Substations (OSM)",
    color="#3498db",
    radius=6,
    popup=["name", "voltage"],
)

# 4. OSM power plants
m.add_geojson(
    plants,
    name="Power plants (OSM)",
    color="#2ecc71",
    radius=8,
    popup=["name", "plant:source", "generator:source"],
)

# 5. OSM data centers
m.add_geojson(
    datacenters,
    name="Data centers (OSM)",
    color="#9b59b6",
    radius=8,
    popup=["name", "operator"],
)

# Render as standalone HTML in output and save a copy to disk

# 6. Urban centers (population >= 100k) as orange markers

# 6. Urban centers (population >= 100k) as orange circles
m.add_geojson(
    urban_centers,
    name="Urban centers (100k+)",

# 7. Land parcels >= 10 ha as green-outlined polygons
if land_parcels["features"]:
    m.add_geojson(
        land_parcels,
        name="Land parcels (>=10ha)",

# 8. Exclusion zones (Natura 2000, flood, nature reserves) as red hatched polygons
if exclusion_zones["features"]:
    m.add_geojson(
        exclusion_zones,
        name="Exclusion zones (fatal flaws)",
        color="#e74c3c",
        fill_color="#e74c3c",
        fill_opacity=0.1,
        weight=1,
        popup=["source", "name", "hazard_type"],
    )
        color="#27ae60",
        fill_color="#27ae60",
        fill_opacity=0.15,
        weight=1,
        popup=["area_ha"],
    )
    color="#f39c12",
    fill_color="#f39c12",
    fill_opacity=0.5,
    radius=12,
    popup=["name", "population"],
)
html_map = m.to_html(width="100%", height="700px")
display(HTML(html_map))

html_path = Path("./data/combined_geolibre_map.html")
html_path.write_text(html_map, encoding="utf-8")
print(f"Saved standalone map HTML to {html_path.resolve()}")

Saved standalone map HTML to C:\Users\mahmoudke\Geo\geotask\project\notebooks\data\combined_geolibre_map.html
